In [ ]:
import from __future__ import annotations

from dataclasses import dataclass, field
import gc
import json
import logging
from pathlib import Path
from typing import Any
from uuid import uuid4

from datasets import load_dataset
from peft import PeftModel
import torch
from transformers.trainer_utils import get_last_checkpoint

from Datasets.dataset_classifier import DatasetClassifier
from Datasets.dataset_request import RequestDataset
from Models.lora import LoRASettings
from Models.model_classifier import Classifier
from Models.models import PPORewardModelProtocol
from Factory import *
import Models.model_evaluator as model_evaluator
from Models.model_policy import PolicyModel
from Models.model_reward import RewardModel
from Models.model_value import ValueModel
from Trainers.trainer_classifier import (
    ClassifierTrainer,
    ClassifierTrainingConfig,
)
from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
import functions

In [ ]:
@dataclass(frozen=True)
class PolicySpec:
    class_name: str = "PolicyModel"
    model_name: str = "Qwen/Qwen3-0.6B"
    lora_config: LoRASettings | None = field(default_factory=LoRASettings)
    checkpoint: str | Path | None = None


@dataclass(frozen=True)
class RewardSpec:
    class_name: str = "RewardModel"
    model_name: str = "Skywork/Skywork-Reward-V2-Qwen3-0.6B"
    mode_name: str = "proxy"


@dataclass(frozen=True)
class DatasetSpec:
    class_name: str = "RequestDataset"
    dataset_name: str = "Anthropic/hh-rlhf"
    start: int = 0
    end: int = 100
    
    
@dataclass(frozen=True)
class EvaluatorSpec:
    class_name: str = "PrometheusEvaluator"
    model_name: str = "prometheus-eval/prometheus-7b-v2.0"


@dataclass(frozen=True)
class TrainingPPOConfig:
    policy: PolicySpec = field(default_factory=PolicySpec)
    reward: RewardSpec = field(default_factory=RewardSpec)
    dataset: DatasetSpec = field(default_factory=DatasetSpec)
    output_dir: str | Path | None = None
    epochs: float = 1.0
    batch_size: int = 8
    gradient_accumulation_steps: int = 8
    rollout_forward_batch_size: int = 16
    response_length: int = 128
    generation_batch_size: int = 64
    num_ppo_epochs: int = 4
    num_mini_batches: int = 1
    learning_rate: float = 3e-6
    save_steps: int = 10
    save_total_limit: int = 1
    logging_steps: int = 10
    
    
@dataclass(frozen=True)
class EvaluateConfig:
    policy: PolicySpec
    evaluator: EvaluatorSpec = field(
        default_factory=lambda: EvaluatorSpec(
            class_name="PrometheusEvaluator",
            model_name="prometheus-eval/prometheus-7b-v2.0",
        )
    )
    evaluator_batch_size: int = 1
    evaluator_reset: bool = False

In [ ]:
from pathlib import Path

policy_spec = PolicySpec()

reward_spec = RewardSpec(
    class_name="DeterministicReward",
    model_name="Qwen/Qwen3-0.6B",
)

dataset_spec = DatasetSpec(end=200)

ppo_config = TrainingPPOConfig(
    policy=policy_spec,
    reward=reward_spec,
    dataset=dataset_spec,
    output_dir="outputs/my_ppo_run",
)

trained_policy = functions.temp_ppo_train_policy(ppo_config)

saved_path = Path(ppo_config.output_dir) / "final"